In [42]:
from transcoder_circuits.circuit_analysis import make_sae_feature_vector, greedy_get_top_paths, print_all_paths, get_paths_via_filter, FeatureFilter, FeatureType, FilterType, paths_to_graph, add_error_nodes_to_graph, ComponentType
from transcoder_circuits.feature_dashboards import get_deembeddings_for_feature_vector, get_deembeddings_for_transcoder_feature, get_transcoder_pullback_features, get_all_feature_scores, get_feature_scores
from sae_training.sparse_autoencoder import SparseAutoencoder
from transformer_lens import HookedTransformer, utils
import torch
import numpy as np
from tqdm import tqdm




def find_top_k_features(model, transcoder, token_ids, top_k=10):
    score = get_all_feature_scores(model, transcoder, token_ids, batch_size=128)

    value, index = torch.topk(score, top_k, dim=-1)

    return value, index

In [43]:
model = HookedTransformer.from_pretrained('gpt2')












from datasets import load_dataset
from utils import tokenize_and_concatenate

dataset = load_dataset('Skylion007/openwebtext', split='train', streaming=True)
dataset = dataset.shuffle(seed=42, buffer_size=10_000)
tokenized_owt = tokenize_and_concatenate(dataset, model.tokenizer, max_length=128, streaming=True)
tokenized_owt = tokenized_owt.shuffle(42)
tokenized_owt = tokenized_owt.take(12800*2)
owt_tokens = np.stack([x['tokens'] for x in tokenized_owt])
owt_tokens_torch = torch.from_numpy(owt_tokens).cuda()

/home/shuhewang/miniconda3/envs/transcoder/lib/python3.10/site-packages/huggingface_hub/file_download.py:797: FutureWarning:

`resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.



Loaded pretrained model gpt2 into HookedTransformer


Token indices sequence length is longer than the specified maximum sequence length for this model (73252 > 1024). Running this sequence through the model will result in indexing errors


In [44]:
transcoder_template = "/data/projects/punim2522/models/gpt2-transcoders/final_sparse_autoencoder_gpt2-small_blocks.{}.ln2.hook_normalized_24576"
transcoders = []
for i in range(12):
    transcoders.append(SparseAutoencoder.load_from_pretrained(f"{transcoder_template.format(i)}.pt").eval())


prompt = "In the hotel laundry room, Emma burned Mary's shirt, so the manager scolded Emma"
# prompt = "Oh, that rifle model is a 6M"
token_strs = model.to_str_tokens(prompt)

print("token_strs", list(enumerate(token_strs)))

token_strs [(0, '<|endoftext|>'), (1, 'In'), (2, ' the'), (3, ' hotel'), (4, ' laundry'), (5, ' room'), (6, ','), (7, ' Emma'), (8, ' burned'), (9, ' Mary'), (10, "'s"), (11, ' shirt'), (12, ','), (13, ' so'), (14, ' the'), (15, ' manager'), (16, ' sc'), (17, 'olded'), (18, ' Emma')]


In [45]:
token_ids = model.tokenizer(prompt, return_tensors='pt').input_ids

print("token_ids", token_ids)

token_ids tensor([[  818,   262,  7541, 25724,  2119,    11, 18966, 11544,  5335,   338,
         10147,    11,   523,   262,  4706,   629, 48959, 18966]])


In [46]:
features, indices = find_top_k_features(model, transcoders[11], token_ids, top_k=10)

print("features.shape:", features.shape)
print("indices.shape:", indices.shape)

features.shape: torch.Size([18, 10])
indices.shape: torch.Size([18, 10])


In [53]:
feature_vector = make_sae_feature_vector(transcoders[11], indices[-2][0], token=-2)
print(feature_vector)
print(feature_vector.vector.shape)

print("indices[idx_][0]:", indices[-2])
print("features[idx_][0]:", features[-2])

mlp11tc[10519]@-2
torch.Size([768])
indices[idx_][0]: tensor([10519, 10564, 12610, 12963,  2270,  6460, 10714, 23082,  9623, 13918],
       device='cuda:0')
features[idx_][0]: tensor([13.0208, 11.1411,  7.1502,  7.1304,  4.4958,  3.6641,  3.5941,  3.5090,
         2.8657,  2.6165], device='cuda:0')


In [51]:
pulledback_feature, deembeddings = get_deembeddings_for_feature_vector(model, feature_vector, k=7)
deembeddings = list(deembeddings)



positive_deembeddings = [(d[0], d[1]) for d in deembeddings]
negative_deembeddings = [(d[2], d[3]) for d in deembeddings]

print("positive_deembeddings:", positive_deembeddings)
print("negative_deembeddings:", negative_deembeddings)

positive_deembeddings: [(6.6762266, ' suspic'), (6.127624, ' advoc'), (6.0795875, ' horizont'), (5.7405634, ' toget'), (5.708954, 'enegger'), (5.616101, ' fortun'), (5.5847683, ' tremend')]
negative_deembeddings: [(-2.9677534, 'uction'), (-2.8307335, 'Shop'), (-2.7995558, 'itect'), (-2.7810633, ' Warehouse'), (-2.7206583, 'fighting'), (-2.7053452, 'topic'), (-2.691846, 'ements')]


In [52]:
from transcoder_circuits.feature_dashboards import display_activating_examples_dash

torch.cuda.empty_cache()

cur_scores = get_feature_scores(model, transcoders[11], owt_tokens_torch[:128*30].cpu(), 18786, batch_size=128, use_raw_scores=False)

cur_scores.shape

100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 30/30 [00:12<00:00,  2.50it/s]


torch.Size([3840, 128])

In [53]:
cur_scores[1:10]

tensor([[0., 0., 0.,  ..., 0., 0., 0.],
        [0., 0., 0.,  ..., 0., 0., 0.],
        [0., 0., 0.,  ..., 0., 0., 0.],
        ...,
        [0., 0., 0.,  ..., 0., 0., 0.],
        [0., 0., 0.,  ..., 0., 0., 0.],
        [0., 0., 0.,  ..., 0., 0., 0.]], device='cuda:0')

In [54]:
display_activating_examples_dash(model, owt_tokens_torch, cur_scores.cpu().numpy(), header_level=None) # don't show dashboard with html headers